In [1]:
import pandas as pd    

df = pd.read_csv('timeseries/datasets/nbabig/csv/draft_history.csv') 

In [2]:
df.head()

,person_id,player_name,season,round_number,round_pick,overall_pick,draft_type,team_id,team_city,team_name,team_abbreviation,organization,organization_type,player_profile_flag
0,79299,Clifton McNeeley,1947,1,1,1,Draft,1610610031,Pittsburgh,Ironmen,PIT,Texas-El Paso,College/University,0
1,78109,Glen Selbo,1947,1,2,2,Draft,1610610035,Toronto,Huskies,HUS,Wisconsin,College/University,1
2,76649,Eddie Ehlers,1947,1,3,3,Draft,1610612738,Boston,Celtics,BOS,Purdue,College/University,1
3,79302,Walt Dropo,1947,1,4,4,Draft,1610610032,Providence,Steamrollers,PRO,Connecticut,College/University,0
4,77048,Dick Holub,1947,1,5,5,Draft,1610612752,New York,Knicks,NYK,Long Island-Brooklyn,College/University,1


In [3]:
df.info()   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7990 entries, 0 to 7989
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   person_id            7990 non-null   int64 
 1   player_name          7990 non-null   object
 2   season               7990 non-null   int64 
 3   round_number         7990 non-null   int64 
 4   round_pick           7990 non-null   int64 
 5   overall_pick         7990 non-null   int64 
 6   draft_type           7990 non-null   object
 7   team_id              7990 non-null   int64 
 8   team_city            7990 non-null   object
 9   team_name            7990 non-null   object
 10  team_abbreviation    7990 non-null   object
 11  organization         7971 non-null   object
 12  organization_type    7971 non-null   object
 13  player_profile_flag  7990 non-null   int64 
dtypes: int64(7), object(7)
memory usage: 874.0+ KB


In [4]:
#lets clean up unnecessary columns
# Drop the unnecessary columns
columns_to_drop = ['person_id', 'round_number', 'round_pick', 'draft_type', 'team_id', 'team_city', 'organization_type', 'player_profile_flag']
df_cleaned = df.drop(columns=columns_to_drop)

# Display the cleaned dataframe
print(df_cleaned.head())


        player_name  season  overall_pick     team_name team_abbreviation  \
0  Clifton McNeeley    1947             1       Ironmen               PIT   
1        Glen Selbo    1947             2       Huskies               HUS   
2      Eddie Ehlers    1947             3       Celtics               BOS   
3        Walt Dropo    1947             4  Steamrollers               PRO   
4        Dick Holub    1947             5        Knicks               NYK   

           organization  
0         Texas-El Paso  
1             Wisconsin  
2                Purdue  
3           Connecticut  
4  Long Island-Brooklyn  


In [5]:
#lets see which organization has the most draft picks
import plotly.express as px

# Group by 'organization' and count the number of players in each organization
organization_counts = df_cleaned.groupby('organization').size().reset_index(name='player_count')

# Sort by player_count in descending order
organization_counts_sorted = organization_counts.sort_values(by='player_count', ascending=False)

# Create an interactive bar plot
fig = px.bar(organization_counts_sorted, 
             x='organization', 
             y='player_count', 
             labels={'organization': 'Organization', 'player_count': 'Number of Players'},
             title='Number of Players per Organization',
             color='player_count',  # Color the bars by the player count
             color_continuous_scale='Viridis')  # Use a continuous color scale

# Display the plot
fig.show()



In [6]:
#lets see top 10 of each organization

import plotly.express as px

# Group by 'organization' and get the top 10 overall_pick for each organization
top_10_overall_pick = df_cleaned.groupby('organization').apply(lambda x: x.nsmallest(10, 'overall_pick')).reset_index(drop=True)

# Create an interactive bar plot to visualize the top 10 overall picks by organization
fig = px.bar(top_10_overall_pick, 
             x='player_name', 
             y='overall_pick', 
             color='organization',  # Color the bars by organization
             labels={'player_name': 'Player Name', 'overall_pick': 'Overall Pick'},
             title='Top 10 Players by Overall Pick for Each Organization',
             color_continuous_scale='Viridis',  # Use a continuous color scale
             category_orders={'organization': top_10_overall_pick['organization'].unique()}  # Order by the organizations
            )

# Display the plot
fig.show()


In [7]:
#1#pick by organization
import plotly.express as px

# Filter the dataset for number 1 overall picks
num_1_picks = df_cleaned[df_cleaned['overall_pick'] == 1]

# Create a count of players from each organization (one player per entry)
fig = px.bar(num_1_picks, 
             x='organization', 
             color='organization', 
             title='Number 1 Overall Pick Players by Organization',
             labels={'organization': 'Organization'},
             text='player_name',  # Display player names on the bars
             category_orders={'organization': num_1_picks['organization'].value_counts().index.tolist()})

# Show the figure
fig.show()


In [8]:
#1#pick by team
import plotly.express as px

# Filter the dataset for number 1 overall picks
num_1_picks = df_cleaned[df_cleaned['overall_pick'] == 1]

# Create a count of players from each organization (one player per entry)
fig = px.bar(num_1_picks, 
             x='team_name', 
             color='organization', 
             title='Number 1 Overall Pick Players by team',
             labels={'team_name': 'team_name'},
             text='player_name',  # Display player names on the bars
             category_orders={'team_name': num_1_picks['team_name'].value_counts().index.tolist()})

# Show the figure
fig.show()


In [9]:
import plotly.graph_objects as go
import pandas as pd

# Drop rows where overall_pick is 0
df_cleaned = df_cleaned[df_cleaned['overall_pick'] > 0]

# Group by season and drop rows with zero picks
season_picks = df_cleaned.groupby('season')['overall_pick'].count().reset_index(name='total_picks')

# Create a dropdown for selecting season
dropdown_buttons = [
    {
        'label': str(season),
        'method': 'update',
        'args': [
            {'x': [df_cleaned[df_cleaned['season'] == season]['player_name']],
             'y': [df_cleaned[df_cleaned['season'] == season]['overall_pick']],
             'marker.color': [df_cleaned[df_cleaned['season'] == season]['overall_pick']],  # Color by overall_pick
             'marker.colorscale': 'Blues',  # Apply a colorscale
             'marker.showscale': True  # Show color scale
            },
            {'title': f'Overall Picks in Season {season}'}
        ]
    } for season in season_picks['season']
]

# Initial plot for the first season
initial_season = season_picks['season'].iloc[0]
initial_data = df_cleaned[df_cleaned['season'] == initial_season]

# Create initial plot
fig = go.Figure(
    data=[go.Bar(
        x=initial_data['player_name'],
        y=initial_data['overall_pick'],
        marker=dict(
            color=initial_data['overall_pick'],  # Color by overall_pick
            colorscale='Blues',  # Apply a colorscale
            showscale=True  # Show color scale
        )
    )],
    layout=go.Layout(
        title=f'Overall Picks in Season {initial_season}',
        updatemenus=[dict(
            type="dropdown",
            x=1.05,  # Move dropdown to the right
            xanchor="left",  # Anchor it to the left side
            y=0.5,  # Positioning vertically at the middle
            yanchor="middle",  # Anchor dropdown at the middle
            buttons=dropdown_buttons
        )],
        height=600,
        width=1000,
        margin=dict(b=100, r=150),  # Add margin on the right to avoid overlap
    )
)

# Show the plot
fig.show()
